In [1]:
from vpython import *
import numpy as np

# ============================================
# ROLLER COASTER ENERGY ANALYZER
# Physics Simulation with VPython
# ============================================

# Simulation parameters
dt = 0.01  # time step
g = 9.8  # gravity (m/s^2)
mass = 100  # mass of cart (kg)
mu = 0.05  # coefficient of friction
track_length = 50  # length of track in x-direction

# Create the scene
scene = canvas(title='Roller Coaster Energy Analyzer',
               width=1200, height=600,
               center=vector(25, 15, 0),
               background=color.white,
               range=35)

# Create the track (a curved path)
def track_function(x):
    """Define the track shape - a roller coaster with hills"""
    return 15 + 8*np.sin(0.2*x) + 3*np.sin(0.5*x) - 0.1*x

# Create visual representation of track
track_objects = []

# Create the rails (two parallel lines)
for x in np.arange(0, track_length, 0.5):
    y = track_function(x)
    # Left rail
    sphere(pos=vector(x, y, -1), radius=0.15, color=color.gray(0.5))
    # Right rail
    sphere(pos=vector(x, y, 1), radius=0.15, color=color.gray(0.5))
    
    # Add cross ties every 2 meters
    if x % 2 < 0.25:
        box(pos=vector(x, y-0.2, 0), 
            size=vector(0.3, 0.1, 2.2), 
            color=color.gray(0.3))

# Create support pillars
for x in np.arange(2, track_length, 5):
    y = track_function(x)
    if y > 5:  # Only add pillars for elevated sections
        pillar = cylinder(pos=vector(x, 0, 0),
                          axis=vector(0, y-0.5, 0),
                          radius=0.3,
                          color=color.gray(0.7))
        # Add a base
        box(pos=vector(x, 0.2, 0), 
            size=vector(1, 0.4, 1), 
            color=color.gray(0.5))

# Add a ground plane
ground = box(pos=vector(track_length/2, -0.5, 0),
             size=vector(track_length+10, 0.5, 20),
             color=color.green)

# Create the cart - make it more visible
cart_body = box(pos=vector(0, track_function(0)+0.8, 0),
                size=vector(2.0, 1.0, 1.5),
                color=color.red)

# Add a roof
cart_roof = pyramid(pos=vector(0, track_function(0)+1.3, 0),
                    size=vector(1.8, 0.4, 1.3),
                    color=color.blue)

# Add wheels
wheel_radius = 0.3
wheel_positions = [
    (-0.7, -0.4, 0.8),  # front right
    (0.7, -0.4, 0.8),   # rear right
    (-0.7, -0.4, -0.8), # front left
    (0.7, -0.4, -0.8)   # rear left
]

wheels = []
for wx, wy, wz in wheel_positions:
    wheel = cylinder(pos=vector(wx, track_function(0)+wy, wz),
                     axis=vector(0, 0, 0.1),
                     radius=wheel_radius,
                     color=color.black)
    wheels.append(wheel)

# Group all cart parts
cart_parts = [cart_body, cart_roof] + wheels

# Create energy display in top-left corner
energy_display = label(pos=vector(-15, 25, 0),
                       text='',
                       xoffset=0, 
                       yoffset=0,
                       height=14, 
                       color=color.black,
                       line=False, 
                       box=False,
                       border=4,
                       font='monospace')

# Create a trail for the cart
trail = []
trail_length = 200

# Physics variables
x = 0.0  # starting position
v = 12.0  # initial velocity (m/s)
t = 0.0
total_work_friction = 0

print("Starting Roller Coaster Simulation...")
print(f"Initial Position: x=0, y={track_function(0):.1f}m")
print(f"Initial Velocity: {v} m/s")
print("Watch the cart move in the VPython window!")
print("Press Ctrl+C in the terminal to stop early")

try:
    while x < track_length - 2 and v > 0.1:  # Stop before the end or if cart almost stops
        rate(60)  # Control animation speed
        
        # Get current y position and slope
        y = track_function(x)
        
        # Calculate slope using numerical derivative
        dx_small = 0.01
        slope = (track_function(x + dx_small) - track_function(x - dx_small)) / (2 * dx_small)
        theta = np.arctan(slope)
        
        # Calculate rotation angle for wheels
        rotation_angle = (v * dt) / wheel_radius
        
        # Update cart position
        cart_body.pos = vector(x, y + 0.8, 0)
        cart_roof.pos = vector(x, y + 1.3, 0)
        
        # Update wheel positions and rotate them
        for i, wheel in enumerate(wheels):
            wx, wy, wz = wheel_positions[i]
            wheel.pos = vector(x + wx, y + 0.8 + wy, wz)
            wheel.rotate(angle=rotation_angle, axis=vector(0, 0, 1))
        
        # Rotate cart to match track slope
        cart_body.axis = vector(1, slope, 0)
        cart_body.up = vector(0, 1, 0)
        cart_roof.axis = vector(1, slope, 0)
        cart_roof.up = vector(0, 1, 0)
        
        # Add point to trail
        trail_sphere = sphere(pos=vector(x, y+0.8, 0), 
                              radius=0.15, 
                              color=color.yellow,
                              opacity=0.4)
        trail.append(trail_sphere)
        
        # Remove old trail points
        if len(trail) > trail_length:
            old_sphere = trail.pop(0)
            old_sphere.visible = False
        
        # ========================================
        # PHYSICS CALCULATIONS
        # ========================================
        
        # Forces
        normal = mass * g * abs(np.cos(theta))  # Normal force magnitude
        
        # Friction force
        friction = mu * normal if abs(v) > 0 else 0
        
        # Component of gravity along the track
        gravity_component = mass * g * np.sin(theta)
        
        # Net force along the track
        net_force = -gravity_component - (friction * np.sign(v))
        
        # Acceleration
        a = net_force / mass
        
        # Numerical integration (Euler's method)
        v = v + a * dt
        x = x + v * dt
        t = t + dt
        
        # ========================================
        # ENERGY CALCULATIONS
        # ========================================
        
        # Kinetic Energy
        ke = 0.5 * mass * v**2
        
        # Potential Energy
        pe = mass * g * y
        
        # Total Mechanical Energy
        te = ke + pe
        
        # Work done by friction
        work_friction = abs(friction * v * dt)
        total_work_friction += work_friction
        
        # Update energy display
        energy_display.text = f"""╔══════════════════════════════════╗
║    ROLLER COASTER ENERGY      ║
╠══════════════════════════════════╣
║ KINETIC ENERGY: {ke:8.0f} J    ║
║ POTENTIAL ENERGY: {pe:8.0f} J    ║
║ TOTAL ENERGY: {te:8.0f} J    ║
║ WORK BY FRICTION: {total_work_friction:8.0f} J    ║
╠══════════════════════════════════╣
║ Time: {t:5.1f}s  Position: {x:5.1f}m ║
║ Velocity: {v:5.1f} m/s  Hill: {np.degrees(theta):5.1f}° ║
╚══════════════════════════════════╝"""

except KeyboardInterrupt:
    print("\nSimulation stopped by user")

# Simple final output
print("\n" + "="*40)
print("SIMULATION COMPLETE")
print("="*40)
print(f"Final Position: x={x:.1f}m")
print(f"Final Velocity: {v:.1f} m/s")
print(f"Total Time: {t:.1f} seconds")
print(f"Total Work by Friction: {total_work_friction:.0f} J")

print("\n✅ Roller coaster simulation finished!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Starting Roller Coaster Simulation...
Initial Position: x=0, y=15.0m
Initial Velocity: 12.0 m/s
Watch the cart move in the VPython window!
Press Ctrl+C in the terminal to stop early

SIMULATION COMPLETE
Final Position: x=48.1m
Final Velocity: 12.7 m/s
Total Time: 4.4 seconds
Total Work by Friction: 1520 J

✅ Roller coaster simulation finished!


In [12]:
from vpython import *
import numpy as np

# ============================================
# COMPLETE LOOP-THE-LOOP ROLLER COASTER
# Every part modeled in detail - FIXED VERSION
# ============================================

class CompleteRollerCoaster:
    def __init__(self):
        # Simulation parameters
        self.dt = 0.01
        self.g = 9.8
        self.mass = 500  # kg (heavier train with multiple cars)
        self.mu = 0.02
        
        # Track parameters
        self.loop_radius = 12
        self.loop_height = 24
        self.track_width = 1.5
        self.rail_radius = 0.15
        
        # Train parameters
        self.num_cars = 3
        self.car_length = 2.5
        self.car_height = 1.2
        self.car_width = 1.8
        
        # Physics variables
        self.angle = 0.0
        self.v = 20.0  # Initial velocity
        self.t = 0.0
        self.total_work_friction = 0
        
        # Force variables
        self.normal_force = 0
        self.upstop_force = 0
        self.upstop_engaged = False
        self.centripetal_force = 0
        self.contact_force = 0
        
        # Collections for all objects
        self.track_parts = []
        self.support_parts = []
        self.train_parts = []
        self.wheels = []
        self.upstop_wheels = []
        self.side_friction_wheels = []
        self.safety_parts = []
        self.theming_parts = []
        
        # Store train positions for animation
        self.car_positions = []
        self.wheel_rotation_angles = []
        
        # Setup
        self.setup_scene()
        self.create_complete_track()
        self.create_support_structure()
        self.create_complete_train()
        self.create_safety_systems()
        self.create_theming()
        self.create_force_arrows()
        self.create_display()
        self.calculate_min_speeds()
    
    def setup_scene(self):
        """Setup the VPython scene with enhanced viewing - FIXED lighting"""
        self.scene = canvas(title='Complete Loop-the-Loop Roller Coaster',
                           width=1400, height=800,
                           center=vector(0, self.loop_height/2 + 5, 0),
                           background=color.white,
                           range=self.loop_height + 15,
                           userspin=True,
                           userpan=True)
        
        # Better camera angle
        self.scene.forward = vector(-1.5, -0.3, -1)
        self.scene.up = vector(0, 1, 0)
        
        # Lighting - FIXED: ambient needs 3 components (RGB)
        self.scene.ambient = vector(0.4, 0.4, 0.4)  # Gray ambient light
        
        # Clear default lights and add custom ones
        self.scene.lights = []
        # Main directional light from top-right
        distant_light(direction=vector(1, 1, 1), color=color.gray(0.8))
        # Fill light from left
        distant_light(direction=vector(-1, 0.5, 0), color=color.gray(0.5))
        # Back light for rim lighting
        distant_light(direction=vector(0, 0, -1), color=color.gray(0.3))
    
    def circular_track_pos(self, angle):
        """Get position on the clothoid-like loop"""
        # Use a slightly elongated shape for better physics
        x = self.loop_radius * cos(angle)
        y = self.loop_radius * sin(angle) * 1.2 + self.loop_radius + 8
        return vector(x, y, 0)
    
    def get_track_orientation(self, angle):
        """Get orientation vectors at a point on the track"""
        tangent = vector(-sin(angle), 1.2 * cos(angle), 0)
        if mag(tangent) > 0:
            tangent = norm(tangent)
        else:
            tangent = vector(0, 1, 0)
        
        right = norm(cross(tangent, vector(0, 0, 1)))
        up = norm(cross(right, tangent))
        return tangent, right, up
    
    def create_complete_track(self):
        """Create detailed track with all rail types"""
        num_segments = 200
        
        # Color scheme for different rail types
        rail_colors = {
            'main': color.gray(0.7),
            'upstop': color.red,
            'side': color.blue,
            'brake': color.orange
        }
        
        for i in range(num_segments + 1):
            angle = 2 * pi * i / num_segments
            pos = self.circular_track_pos(angle)
            
            # Get track orientation
            tangent, right, up = self.get_track_orientation(angle)
            
            # MAIN RUNNING RAILS (top)
            rail_offset = right * self.track_width/2
            
            # Left main rail - use cylinders for smoother look
            if i < num_segments:
                next_angle = 2 * pi * (i + 1) / num_segments
                next_pos = self.circular_track_pos(next_angle)
                
                # Left main rail segment
                cylinder(pos=pos + rail_offset + up * 0.3,
                        axis=next_pos - pos,
                        radius=self.rail_radius,
                        color=rail_colors['main'])
                
                # Right main rail segment
                cylinder(pos=pos - rail_offset + up * 0.3,
                        axis=next_pos - pos,
                        radius=self.rail_radius,
                        color=rail_colors['main'])
                
                # UP-STOP RAILS (bottom - prevent lifting)
                cylinder(pos=pos + rail_offset - up * 0.3,
                        axis=next_pos - pos,
                        radius=self.rail_radius * 0.8,
                        color=rail_colors['upstop'])
                
                cylinder(pos=pos - rail_offset - up * 0.3,
                        axis=next_pos - pos,
                        radius=self.rail_radius * 0.8,
                        color=rail_colors['upstop'])
            
            # CROSS TIES (horizontal supports between rails)
            if i % 4 == 0:
                tie = box(pos=pos + up * 0.1,
                         size=vector(0.1, 0.1, self.track_width + 0.4),
                         color=color.gray(0.5))
                self.track_parts.append(tie)
            
            # SPINE (central support beam)
            if i % 2 == 0:
                if i < num_segments:
                    next_pos = self.circular_track_pos(angle + 2*pi/num_segments)
                    spine_seg = cylinder(pos=pos - up * 0.5,
                                        axis=next_pos - pos,
                                        radius=0.2,
                                        color=color.gray(0.4))
                    self.track_parts.append(spine_seg)
            
            # RAIL JOINTS (every 10 segments)
            if i % 10 == 0:
                joint = box(pos=pos + up * 0.3,
                           size=vector(0.3, 0.1, self.track_width + 0.2),
                           color=color.gray(0.3))
                self.track_parts.append(joint)
        
        print("✓ Track system complete")
    
    def create_support_structure(self):
        """Create comprehensive support structure"""
        # MAIN SUPPORT PILLARS
        for angle in np.linspace(0, 2*pi, 16):
            pos = self.circular_track_pos(angle)
            
            # Only add pillars where track is elevated
            if pos.y > 3:
                # Main pillar
                pillar_height = pos.y - 1
                pillar = cylinder(pos=vector(pos.x, 1, pos.z),
                                 axis=vector(0, pillar_height, 0),
                                 radius=0.4,
                                 color=color.gray(0.6))
                self.support_parts.append(pillar)
                
                # FOOTERS (concrete bases)
                footer = box(pos=vector(pos.x, 0.5, pos.z),
                            size=vector(1.2, 1, 1.2),
                            color=color.gray(0.3))
                self.support_parts.append(footer)
        
        # Add diagonal bracing between pillars
        for i in range(0, len(self.support_parts), 4):
            if i + 4 < len(self.support_parts):
                p1 = self.support_parts[i].pos
                p2 = self.support_parts[i+4].pos
                if isinstance(p1, vector) and isinstance(p2, vector):
                    brace = cylinder(pos=p1 + vector(0, p1.y, 0),
                                    axis=p2 - p1,
                                    radius=0.15,
                                    color=color.gray(0.5))
                    self.support_parts.append(brace)
        
        # GROUND FOUNDATION
        ground = box(pos=vector(0, -0.5, 0),
                    size=vector(60, 1, 40),
                    color=color.green)
        self.support_parts.append(ground)
        
        # CONCRETE FOOTERS at key points
        for x in [-20, -10, 0, 10, 20]:
            footer = box(pos=vector(x, 0, 15),
                        size=vector(3, 1, 3),
                        color=color.gray(0.4))
            self.support_parts.append(footer)
        
        print("✓ Support structure complete")
    
    def create_complete_train(self):
        """Create a detailed train with all wheel types"""
        start_pos = self.circular_track_pos(0)
        
        # Create multiple cars
        for car_num in range(self.num_cars):
            car_offset = car_num * self.car_length * 1.1
            
            # CHASSIS (main frame)
            chassis = box(pos=start_pos + vector(-car_offset, 0.5, 0),
                         size=vector(self.car_length, 0.3, self.car_width),
                         color=color.gray(0.2))
            self.train_parts.append(chassis)
            self.car_positions.append(chassis)
            
            # CAR BODY (passenger compartment)
            body = box(pos=start_pos + vector(-car_offset, 1.0, 0),
                      size=vector(self.car_length-0.4, 0.8, self.car_width-0.3),
                      color=color.red)
            self.train_parts.append(body)
            
            # ROOF
            roof = pyramid(pos=start_pos + vector(-car_offset, 1.5, 0),
                          size=vector(self.car_length-0.6, 0.3, self.car_width-0.5),
                          color=color.blue,
                          opacity=0.9)
            self.train_parts.append(roof)
            
            # PASSENGERS (simple representations)
            for seat in range(2):
                passenger_pos = start_pos + vector(-car_offset - 0.5 + seat*1.0, 1.2, 0.4)
                # Head
                sphere(pos=passenger_pos + vector(0, 0.2, 0),
                      radius=0.15,
                      color=color.yellow)
                # Body
                cylinder(pos=passenger_pos - vector(0, 0.3, 0),
                        axis=vector(0, 0.6, 0),
                        radius=0.1,
                        color=color.orange)
            
            # WHEEL BOGIES (wheel assemblies)
            for bogie_pos in [-0.8, 0.8]:
                bogie = box(pos=start_pos + vector(-car_offset + bogie_pos, -0.2, 0),
                           size=vector(0.6, 0.2, self.car_width-0.2),
                           color=color.gray(0.3))
                self.train_parts.append(bogie)
                
                # ROAD WHEELS (main wheels - on top of rail)
                for side in [-0.6, 0.6]:
                    wheel = cylinder(pos=start_pos + vector(-car_offset + bogie_pos, 0, side),
                                    axis=vector(0, 0, 0.1),
                                    radius=0.25,
                                    color=color.black)
                    self.wheels.append(wheel)
                    self.wheel_rotation_angles.append(0)
                
                # UP-STOP WHEELS (under the track)
                for side in [-0.5, 0.5]:
                    upstop = cylinder(pos=start_pos + vector(-car_offset + bogie_pos, 0.8, side),
                                     axis=vector(0, 0, 0.1),
                                     radius=0.18,
                                     color=color.yellow)
                    self.upstop_wheels.append(upstop)
                
                # SIDE FRICTION WHEELS (keep centered)
                for side in [-0.9, 0.9]:
                    side_wheel = sphere(pos=start_pos + vector(-car_offset + bogie_pos, 0.4, side),
                                       radius=0.15,
                                       color=color.cyan)
                    self.side_friction_wheels.append(side_wheel)
            
            # BRAKE FINS (for stopping)
            brake_fin = box(pos=start_pos + vector(-car_offset + 1, 0.3, 0),
                           size=vector(0.1, 0.4, 0.6),
                           color=color.red)
            self.safety_parts.append(brake_fin)
        
        # ANTI-ROLLBACK DEVICE at start
        anti_roll = box(pos=start_pos + vector(-1, -0.3, 0),
                       size=vector(0.4, 0.2, 0.8),
                       color=color.orange)
        self.safety_parts.append(anti_roll)
        
        print("✓ Complete train with", self.num_cars, "cars assembled")
    
    def create_safety_systems(self):
        """Add safety systems along the track"""
        
        # PROXIMITY SENSORS (along track)
        for angle in np.linspace(0, 2*pi, 20):
            pos = self.circular_track_pos(angle)
            sensor = box(pos=pos + vector(0, 0, 2),
                        size=vector(0.2, 0.2, 0.5),
                        color=color.green)
            self.safety_parts.append(sensor)
        
        # BLOCK BRAKES (sections of track that can brake)
        for angle in [0, pi/2, pi, 3*pi/2]:
            pos = self.circular_track_pos(angle)
            brake_section = box(pos=pos + vector(0, -0.5, 0),
                               size=vector(2, 0.1, 2),
                               color=color.red,
                               opacity=0.5)
            self.safety_parts.append(brake_section)
        
        # STATION BRAKES (loading area)
        station_pos = self.circular_track_pos(0)
        station_brake = box(pos=station_pos + vector(-5, -0.3, 0),
                           size=vector(3, 0.2, 1.5),
                           color=color.orange)
        self.safety_parts.append(station_brake)
        
        print("✓ Safety systems installed")
    
    def create_theming(self):
        """Add decorative elements to make it look great"""
        
        # ENTRANCE ARCH
        entrance_pos = self.circular_track_pos(0) + vector(-5, 0, -5)
        arch_left = cylinder(pos=entrance_pos,
                            axis=vector(0, 5, 0),
                            radius=0.3,
                            color=color.blue)
        self.theming_parts.append(arch_left)
        
        arch_right = cylinder(pos=entrance_pos + vector(8, 0, 0),
                             axis=vector(0, 5, 0),
                             radius=0.3,
                             color=color.blue)
        self.theming_parts.append(arch_right)
        
        arch_top = box(pos=entrance_pos + vector(4, 5, 0),
                      size=vector(8, 0.3, 0.5),
                      color=color.blue)
        self.theming_parts.append(arch_top)
        
        # LIGHTS along track
        for angle in np.linspace(0, 2*pi, 30):
            pos = self.circular_track_pos(angle)
            light = sphere(pos=pos + vector(0, 1, 2),
                          radius=0.2,
                          color=color.yellow,
                          emissive=True)
            self.theming_parts.append(light)
        
        # FLAGS at top
        top_pos = self.circular_track_pos(pi/2)
        flag_pole = cylinder(pos=top_pos + vector(2, 0, 2),
                            axis=vector(0, 2, 0),
                            radius=0.1,
                            color=color.white)
        self.theming_parts.append(flag_pole)
        
        flag = box(pos=top_pos + vector(2.5, 2, 2),
                  size=vector(1, 0.6, 0.1),
                  color=color.red)
        self.theming_parts.append(flag)
        
        # CLOUDS (decorative)
        for i in range(5):
            cloud_pos = vector(-15 + i*8, 25, -10)
            for j in range(3):
                sphere(pos=cloud_pos + vector(j*1.5, 0, 0),
                      radius=1.0,
                      color=color.white,
                      opacity=0.3)
        
        # PHOTO booth at exit
        photo_pos = self.circular_track_pos(2*pi) + vector(5, 0, 5)
        booth = box(pos=photo_pos,
                   size=vector(3, 3, 3),
                   color=color.purple)
        self.theming_parts.append(booth)
        
        camera = sphere(pos=photo_pos + vector(0, 1, 2),
                       radius=0.3,
                       color=color.black)
        self.theming_parts.append(camera)
        
        print("✓ Theming complete - looking great!")
    
    def create_force_arrows(self):
        """Create arrows to visualize forces"""
        self.centripetal_arrow = arrow(pos=vector(0,0,0),
                                       axis=vector(0,0,0),
                                       color=color.magenta,
                                       shaftwidth=0.3,
                                       opacity=0.8,
                                       visible=True)
        
        self.force_arrow = arrow(pos=vector(0,0,0),
                                 axis=vector(0,0,0),
                                 color=color.green,
                                 shaftwidth=0.3,
                                 opacity=0.8,
                                 visible=True)
    
    def create_display(self):
        """Create the energy display"""
        self.energy_display = label(pos=vector(0, 0, 0),
                                   text='',
                                   xoffset=20,
                                   yoffset=30,
                                   height=12,
                                   color=color.black,
                                   line=False,
                                   box=True,
                                   border=6,
                                   font='monospace',
                                   pixel_pos=True)
    
    def calculate_min_speeds(self):
        """Calculate minimum speeds needed"""
        self.min_speed_top = sqrt(self.loop_radius * self.g * 0.8)  # Adjusted for clothoid
        self.min_speed_bottom = sqrt(5 * self.loop_radius * self.g * 0.8)
        
        print("\n" + "="*60)
        print("COMPLETE LOOP-THE-LOOP ROLLER COASTER")
        print("="*60)
        print(f"Track Components: Main Rails, Up-Stop Rails, Side Rails")
        print(f"Train: {self.num_cars} cars with all wheel types")
        print(f"Safety Systems: Block brakes, sensors, anti-rollback")
        print(f"Initial velocity: {self.v} m/s")
        print("\n🎢 Ride features:")
        print("  • Complete wheel system (road, up-stop, side friction)")
        print("  • Detailed support structure with footers")
        print("  • Safety systems throughout track")
        print("  • Theming and decorative elements")
        print("  • Force visualization arrows")
        print("\nControls: Drag to rotate | Scroll to zoom")
        print("Press Ctrl+C in terminal to stop\n")
    
    def calculate_physics(self):
        """Calculate all physics parameters"""
        pos = self.circular_track_pos(self.angle)
        
        # Direction to center (approximate for clothoid)
        center_pos = vector(0, self.loop_radius + 8, 0)
        to_center = norm(center_pos - pos)
        
        # Get orientation
        tangent, right, up = self.get_track_orientation(self.angle)
        self.tangent_dir = tangent
        
        # Centripetal force needed
        self.centripetal_accel = self.v**2 / self.loop_radius
        self.centripetal_force = self.mass * self.centripetal_accel
        
        # Gravity components
        gravity_tangential = self.mass * self.g * sin(self.angle) * 0.8
        gravity_radial = self.mass * self.g * cos(self.angle) * 0.8
        
        # Required normal force
        required_normal = self.mass * self.centripetal_accel - gravity_radial
        
        # Check if up-stop needed
        if required_normal < 0:
            self.upstop_engaged = True
            self.upstop_force = -required_normal
            self.normal_force = 0
            self.contact_force = self.upstop_force
            force_direction = -to_center
        else:
            self.upstop_engaged = False
            self.upstop_force = 0
            self.normal_force = required_normal
            self.contact_force = self.normal_force
            force_direction = to_center
        
        # Friction
        friction_source = self.upstop_force if self.upstop_engaged else self.normal_force
        friction = self.mu * abs(friction_source) if abs(self.v) > 0 else 0
        
        # Net force and acceleration
        net_force = -gravity_tangential - (friction * sign(self.v))
        a = net_force / self.mass
        
        # Work by friction
        work_friction = abs(friction * self.v * self.dt)
        self.total_work_friction += work_friction
        
        # Energy
        self.ke = 0.5 * self.mass * self.v**2
        self.pe = self.mass * self.g * pos.y
        self.te = self.ke + self.pe
        
        # Update force arrows
        self.update_force_arrows(pos, to_center, force_direction)
        
        return a, pos
    
    def update_force_arrows(self, pos, to_center, force_direction):
        """Update force visualization"""
        cart_pos = pos + vector(0, 0.8, 0)
        
        # Centripetal arrow (magenta) - points to center
        self.centripetal_arrow.pos = cart_pos
        self.centripetal_arrow.axis = to_center * self.centripetal_force * 0.002
        
        # Contact force arrow
        self.force_arrow.pos = cart_pos
        self.force_arrow.axis = force_direction * self.contact_force * 0.002
        self.force_arrow.color = color.red if self.upstop_engaged else color.green
    
    def update_display(self, pos):
        """Update the display"""
        if self.upstop_engaged:
            force_status = f"UP-STOP: {self.upstop_force/1000:5.1f} kN"
            g_force = self.upstop_force / (self.mass * self.g)
        else:
            force_status = f"NORMAL: {self.normal_force/1000:5.1f} kN"
            g_force = self.normal_force / (self.mass * self.g)
        
        # Status message
        if self.upstop_engaged:
            status = "⚠ Up-stop Active"
        elif self.v < self.min_speed_top and self.angle > pi/2:
            status = "⚠ Too slow!"
        else:
            status = "✓ Normal operation"
        
        self.energy_display.text = f"""╔════════════════════════════════════════╗
║     COMPLETE LOOP ROLLER COASTER     ║
╠════════════════════════════════════════╣
║ ENERGY:                                ║
║  KE: {self.ke:8.0f} J  PE: {self.pe:8.0f} J        ║
║  Total: {self.te:8.0f} J  Friction: {self.total_work_friction:6.0f} J ║
╠════════════════════════════════════════╣
║ MOTION:                                ║
║  Angle: {degrees(self.angle):5.1f}°  Vel: {self.v:5.1f} m/s    ║
║  Position: ({pos.x:5.1f},{pos.y:5.1f},{pos.z:5.1f}) ║
╠════════════════════════════════════════╣
║ FORCES:                                ║
║  Centripetal: {self.centripetal_force:6.0f} N        ║
║  {force_status}  ({g_force:4.1f} G) ║
╠════════════════════════════════════════╣
║ {status:40} ║
╚════════════════════════════════════════╝"""
    
    def update_train(self, pos):
        """Update train position and orientation"""
        # Get orientation
        tangent, right, up = self.get_track_orientation(self.angle)
        
        # Calculate wheel rotation
        wheel_rotate = (self.v * self.dt) / 0.25
        
        # Update wheel colors based on up-stop engagement
        wheel_color = color.red if self.upstop_engaged else color.black
        
        # Update wheels rotation (simplified - just rotate all wheels)
        for i, wheel in enumerate(self.wheels):
            if i < len(self.wheel_rotation_angles):
                self.wheel_rotation_angles[i] += wheel_rotate
                # We can't easily rotate cylinders around their axis in VPython,
                # but we can change color to indicate state
                wheel.color = wheel_color
        
        # Update up-stop wheels color
        for upstop in self.upstop_wheels:
            upstop.color = color.red if self.upstop_engaged else color.yellow
    
    def run(self):
        """Main simulation loop"""
        try:
            while self.angle < 2*pi and self.v > 0.1:
                rate(60)
                
                # Physics
                a, pos = self.calculate_physics()
                
                # Update state
                self.v += a * self.dt
                self.angle += (self.v / self.loop_radius) * self.dt
                self.t += self.dt
                
                # Update visuals
                self.update_train(pos)
                self.update_display(pos)
                
        except KeyboardInterrupt:
            print("\nSimulation stopped by user")
        
        # Final report
        self.show_final_results()
    
    def show_final_results(self):
        """Display final results"""
        print("\n" + "="*60)
        print("RIDE COMPLETE - COMPONENT SUMMARY")
        print("="*60)
        print(f"Track sections: Main, Up-stop, Side friction rails")
        print(f"Support structure: {len(self.support_parts)} components")
        print(f"Train: {self.num_cars} cars with {len(self.wheels)} wheels")
        print(f"Safety systems: {len(self.safety_parts)} devices")
        print(f"Theming elements: {len(self.theming_parts)} decorations")
        print(f"\nRide statistics:")
        print(f"  Final speed: {self.v:.1f} m/s")
        print(f"  Max G-force: {max(abs(self.normal_force), abs(self.upstop_force))/(self.mass*self.g):.1f} G")
        print(f"  Energy lost to friction: {self.total_work_friction:.0f} J")
        
        if self.angle >= 2*pi:
            print("\n✅ SUCCESS! Full loop completed!")
        else:
            print("\n❌ Could not complete full loop")

# ============================================
# RUN THE COMPLETE SIMULATION
# ============================================

if __name__ == "__main__":
    coaster = CompleteRollerCoaster()
    coaster.run()




<IPython.core.display.Javascript object>

✓ Track system complete
✓ Support structure complete
✓ Complete train with 3 cars assembled
✓ Safety systems installed
✓ Theming complete - looking great!

COMPLETE LOOP-THE-LOOP ROLLER COASTER
Track Components: Main Rails, Up-Stop Rails, Side Rails
Train: 3 cars with all wheel types
Safety Systems: Block brakes, sensors, anti-rollback
Initial velocity: 20.0 m/s

🎢 Ride features:
  • Complete wheel system (road, up-stop, side friction)
  • Detailed support structure with footers
  • Safety systems throughout track
  • Theming and decorative elements
  • Force visualization arrows

Controls: Drag to rotate | Scroll to zoom
Press Ctrl+C in terminal to stop


RIDE COMPLETE - COMPONENT SUMMARY
Track sections: Main, Up-stop, Side friction rails
Support structure: 46 components
Train: 3 cars with 12 wheels
Safety systems: 29 devices
Theming elements: 37 decorations

Ride statistics:
  Final speed: 0.1 m/s
  Max G-force: 0.8 G
  Energy lost to friction: 6150 J

❌ Could not complete full loop
